# ParetoBandit Demo Playground

**ParetoBandit** is an adaptive LLM router that uses a *contextual bandit* algorithm
(Disjoint LinUCB) to learn, in real time, which language model gives the best
quality-per-dollar for each incoming prompt (paper §3).

### The core idea

You have access to several LLMs — say, a cheap model (Llama-8B), a mid-tier model
(Mistral-Large), and a premium model (Gemini-Pro). Each request has a *context*
(the embedded prompt, §2.2). The router learns a mapping from context to expected
quality for every model, and picks the one that maximises a budget-augmented UCB
score (Eq. 2) that balances **quality** against **cost**.

### How the operator controls the router

In practice, you set a **budget target** *B* ($/request) and the router handles
the rest. Internally, each routing decision maximises:

> score = *exploit* + α · *explore* − (λ_c + λ_t) · *cost*  (Eq. 2, §3.2)

The key parameters are:

| Parameter | What it controls | Paper ref |
|-----------|-----------------|-----------|
| **Budget target** *B* | The operator's maximum acceptable average cost per request. The BudgetPacer's adaptive multiplier λ_t enforces this in closed loop. | §2.3, §3.2 Eqs. 3–4 |
| `alpha` | **Exploration vs. exploitation.** Width of the UCB confidence bonus. Higher → more exploration. | §3.2 Eq. 2 |
| `forgetting_factor` (γ) | **Adaptation speed.** Geometric discount on sufficient statistics. At γ=0.997 the effective memory is ~333 observations. | §3.3 Eqs. 7–8 |

Under the hood, there is also a **static cost penalty** λ_c (default 0.3) that
encodes a baseline cost–quality preference from the first request (§3.2). In
normal operation you do not need to tune it — the BudgetPacer's adaptive λ_t
provides closed-loop budget enforcement on top of λ_c.

### What this notebook covers

1. Load evaluation data (shipped with the library, §4.1)
2. Run your first routing trial — see model selection fractions
3. Experiment with budget-paced routing (§4.2)
4. Sweep parameters and observe the quality-cost trade-off
5. Run the full pre-built scenarios (§4.2–§4.5)
6. Bring your own data

---
## 1. Setup

This notebook requires the `[demo]` extra, which includes the embedding model
(`sentence-transformers`) and `matplotlib`:

```bash
pip install paretobandit[demo]
```

If you cloned the repo and installed in editable mode (`pip install -e ".[demo]"`),
you are already set.

In [ ]:
!pip install paretobandit[demo]

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pareto_bandit.demo import (
    DemoConfig,
    load_demo_splits,
    run_trial,
    run_scenario_1,
    run_scenario_2,
    run_scenario_3,
    run_scenario_4,
    ARM_ORDER,
    ARM_SHORT,
)
from pareto_bandit.feature_service import FeatureService
from pareto_bandit.budget_pacer import BudgetPacer, PacingMode

print("Imports OK")

---
## 2. Load and Inspect the Evaluation Data

ParetoBandit ships two datasets drawn from nine public benchmarks (GSM8K,
WinoGrande, MMLU, etc.), matching the paper's experimental protocol (§4.1):

- **val.jsonl** (*n*=1,785 prompts) — used for **online learning** (the router
  trains on these).
- **test_holdout.jsonl** (*n*=1,824 prompts) — used for **evaluation** (held-out
  prompts the router has never trained on).

Each prompt has been evaluated against the K=3 model portfolio (Table 1):

| Arm | Typical cost | Role |
|-----|-------------|------|
| **Llama-8B** | ~\$2.9×10⁻⁵ / req | Budget — fast, cheap, good on easy tasks |
| **Mistral-Large** | ~\$5.3×10⁻⁴ / req | Mid-tier — strong generalist |
| **Gemini-Pro** | ~\$1.5×10⁻² / req | Premium — best on hard reasoning, 530× more expensive than Llama |

- **Reward** (0–1): composite quality score from a DeepSeek-R1 judge (§4.1).
- **Cost** (USD): actual API cost for that request.

> **Try it:** Point `val_file` / `holdout_file` at your own JSONL files to use
> custom data.

In [ ]:
fs = FeatureService()
cfg = DemoConfig()

train, holdout = load_demo_splits(
    val_file=cfg.val_file,
    holdout_file=cfg.holdout_file,
    feature_service=fs,
)

print(f"Train (val):     {train.n} prompts")
print(f"Holdout (test):  {holdout.n} prompts")
print(f"Features per prompt: {train.embeddings.shape[1] - 1} + 1 bias\n")

print(f"{'Model':<18s}  {'Avg Reward':>10s}  {'Avg Cost (USD)':>14s}")
print("-" * 46)
for arm in ARM_ORDER:
    r = np.mean(train.rewards[arm])
    c = np.mean(train.costs[arm])
    print(f"{ARM_SHORT[arm]:<18s}  {r:>10.3f}  ${c:>13.6f}")

---
## 3. Your First Routing Trial

A **trial** works in two phases:

1. **Online learning** — the router sees each **training (val)** prompt, picks
   a model, observes the reward and cost, and updates its per-arm sufficient
   statistics via geometric forgetting (§3.3, Eqs. 7–8).
2. **Evaluation** — the router continues routing on **holdout** prompts (still
   learning — standard bandit protocol). We measure average reward, cost,
   and which models were selected.

### Understanding the static cost penalty λ_c

In production you would set a **budget target** *B* ($/req) and let the
BudgetPacer handle cost control adaptively (Section 4 below). Here, to build
intuition, we first show the effect of the **static cost penalty** λ_c
(denoted `cost_penalty` in the API), which is the baseline cost-aversion
weight in the UCB score (§3.2, Eq. 2):

- **`alpha`** (default 0.01): LinUCB exploration coefficient — width of the
  confidence bonus (§3.2).
- **`forgetting_factor`** (default 0.997): geometric discount γ on sufficient
  statistics. At 0.997 the effective memory is ~333 observations (§3.3).
- **`cost_penalty`** λ_c (default 0.3): static cost-aversion weight. At 0.0
  the router ignores cost (quality-only); at 1.0 it strongly prefers cheap
  models. In practice, the BudgetPacer's adaptive λ_t handles this for you.

Below we sweep `cost_penalty` to show how the static weight shapes model
selection — the BudgetPacer automates this in Section 4:

In [ ]:
for cp_label, cp_value in [("quality-only", 0.0), ("balanced", 0.3), ("cost-focused", 1.0)]:
    trial = run_trial(
        train, holdout,
        alpha=0.01,
        forgetting_factor=0.997,
        cost_penalty=cp_value,
        seed=42,
    )
    fracs = ", ".join(
        f"{ARM_SHORT[a]}={trial.model_fractions[a]:.0%}" for a in ARM_ORDER
    )
    print(
        f"cost_penalty={cp_value:.1f} ({cp_label:>13s}):  "
        f"reward={trial.mean_reward:.4f}  "
        f"cost=${trial.mean_cost:.6f}  "
        f"[{fracs}]"
    )

Notice how the static cost penalty λ_c shapes routing:
- At `cost_penalty=0.0` (λ_c=0), the router picks the highest-quality model regardless of price.
- At `cost_penalty=1.0`, it routes almost everything to Llama-8B (cheapest).
- The default (0.3) strikes a balance.

In practice, you rarely need to tune λ_c directly — the **BudgetPacer** (next
section) accepts a dollar budget target and adaptively adjusts its own penalty
λ_t on top of λ_c to enforce it (§3.2, Eqs. 3–4).

> **Try it:** Change `alpha` to 0.001 (greedy) or 0.1 (exploratory) and re-run.
> Or set `forgetting_factor=1.0` to disable forgetting (stationary LinUCB).

---
## 4. Budget-Paced Routing — The Primary Interface

This is the interface you will use in practice. Instead of tuning λ_c manually,
you specify a **budget target** *B* — the maximum acceptable average cost per
request in dollars (§2.3). The **BudgetPacer** (§3.2) then maintains an
adaptive Lagrangian multiplier λ_t that adjusts *every request* via smoothed
dual ascent (Eqs. 3–4):

- If the EMA-smoothed cost is **above** *B* → λ_t increases → expensive
  models are penalised more → traffic shifts to cheaper models.
- If **below** *B* → λ_t decreases → the router is free to pursue quality
  with more expensive models.

This two-layer enforcement — a soft λ_t penalty plus a hard per-request cost
ceiling (Algorithm 1, §3.2) — gives you **budget compliance** without
sacrificing more quality than necessary.

Below, we sweep five budget targets and plot the resulting quality-cost
frontier (compare paper Figure 1, §4.2).

> **Try it:** Edit `budget_targets` to add tighter or looser budgets and see
> how the router adapts.

In [ ]:
budget_targets = np.geomspace(3e-5, 1.5e-2, num=5)

sweep_rewards, sweep_costs = [], []

for target in budget_targets:
    pacer = BudgetPacer(
        target_avg_spend_usd=target,
        mode=PacingMode.ADAPTIVE,
    )
    trial = run_trial(
        train, holdout,
        cost_penalty=0.0,
        budget_pacer=pacer,
        seed=42,
    )
    sweep_rewards.append(trial.mean_reward)
    sweep_costs.append(trial.mean_cost)
    print(f"  target=${target:.2e}  ->  reward={trial.mean_reward:.4f}  cost=${trial.mean_cost:.2e}")

# Plot the Pareto frontier
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_costs, sweep_rewards, "o-", color="#0072B2", linewidth=2, markersize=8)
for t, r, c in zip(budget_targets, sweep_rewards, sweep_costs):
    ax.annotate(f"${t:.1e}", (c, r), textcoords="offset points",
                xytext=(8, -4), fontsize=8, color="0.4")
ax.set_xlabel("Avg Cost per Request (USD)", fontsize=11)
ax.set_ylabel("Mean Reward (Quality)", fontsize=11)
ax.set_xscale("log")
ax.set_title("Budget-Paced Quality vs. Cost Frontier", fontweight="bold")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 5. "What If" Parameter Sweep

Let's systematically sweep one parameter at a time to see how it shapes the
model mix. Below we sweep the static `cost_penalty` λ_c (§3.2, Eq. 2) and
show the resulting model selection fractions alongside reward and cost.

**What to expect:**
- As λ_c increases, the fraction of **Llama-8B** (cheapest) rises
  and **Gemini-Pro** (most expensive) drops.
- Reward decreases because cheaper models are generally lower quality.
- Cost drops sharply.

> **Try it:** Replace `"cost_penalty"` with `"alpha"` (exploration, §3.2) or
> `"forgetting_factor"` (adaptation speed, §3.3) in the code below and adjust
> the values list to explore those dimensions.

In [ ]:
sweep_param = "cost_penalty"
sweep_values = [0.0, 0.1, 0.3, 0.5, 1.0]
defaults = dict(alpha=0.01, forgetting_factor=0.997, cost_penalty=0.3)

print(f"{'Value':>8s}  {'Reward':>7s}  {'Cost (USD)':>11s}  ", end="")
for a in ARM_ORDER:
    print(f"{ARM_SHORT[a]:>14s}", end="")
print()
print("-" * 80)

for val in sweep_values:
    kwargs = {**defaults, sweep_param: val}
    trial = run_trial(train, holdout, seed=42, **kwargs)
    print(f"{val:>8.3f}  {trial.mean_reward:>7.4f}  ${trial.mean_cost:>10.6f}  ", end="")
    for a in ARM_ORDER:
        print(f"{trial.model_fractions[a]:>13.1%}", end=" ")
    print()

---
## 6. Run the Full Built-In Scenarios

The demo ships four pre-built scenarios that produce publication-quality
multi-panel plots. Each uses the paper's **train-on-val / evaluate-on-holdout**
protocol (§4.1) with 10 independent seeds (configurable; paper uses 20).

- **Scenario 1 — Budget-Paced Routing (§4.2, Figure 1):** Quality-cost Pareto
  frontier, budget compliance, and model allocation across 7 budget levels.
- **Scenario 2 — Quality Degradation & Recovery (§4.4, Figure 3):** 3-phase
  (608 prompts/phase): Normal → Mistral-Large failure (reward → 0.75) →
  Recovery. Geometric forgetting (§3.3) detects the regression and shifts
  traffic; Phase 3 reuses Phase 1 prompts for within-subject comparison.
- **Scenario 3 — Cost Drift & Recovery (§4.3, Figure 2):** 3-phase:
  Normal → Gemini-Pro price drop → Price Restored. The BudgetPacer (§3.2)
  exploits cheap premium routing; Phase 3 reverts pricing and reuses Phase 1
  prompts.
- **Scenario 4 — Configuration Comparison:** Side-by-side effect of alpha
  (§3.2), forgetting_factor (§3.3), and cost_penalty λ_c on the model mix.

> **Tip:** From the command line, you can run all scenarios at once:
> ```bash
> paretobandit-demo                    # all 4 scenarios
> paretobandit-demo --scenario 2       # just one
> ```

In [ ]:
from IPython.display import Image, display

cfg = DemoConfig(
    n_seeds=5,
    output_dir="demo_results",
)

out_path = run_scenario_1(cfg, train, holdout)
display(Image(filename=str(out_path)))

> **Try it:** Replace `run_scenario_1` with `run_scenario_2`, `run_scenario_3`,
> or `run_scenario_4` to explore the other scenarios. Increase `cfg.n_seeds`
> (default 10, paper uses 20) for smoother curves, or adjust `cfg.alpha` and
> `cfg.forgetting_factor` to see how results change.

---
## 7. Bring Your Own Data

To use your own evaluation data, prepare **two JSONL files** (one for training,
one for holdout evaluation) where each line is a JSON object with this structure:

```json
{
  "prompt": "What is 2 + 2?",
  "arms": {
    "meta-llama/llama-3.1-8b-instruct": {"reward": 1.0, "cost": 0.00002},
    "mistralai/mistral-large-2512":     {"reward": 1.0, "cost": 0.00030},
    "google/gemini-2.5-pro":            {"reward": 1.0, "cost": 0.01000}
  }
}
```

Each record must include all three arm IDs with a `reward` (0–1) and `cost`
(USD) per arm. Rewards should be continuous scores in [0, 1] (§4.1).

### Using a custom encoder

By default, prompts are embedded with `all-MiniLM-L6-v2` + PCA to 25 dims
(§2.2). To use a different SentenceTransformer model, create a custom
`FeatureService`:

```python
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-mpnet-base-v2")
custom_fs = FeatureService(
    custom_encoder=lambda text: st_model.encode(
        text, normalize_embeddings=True, show_progress_bar=False
    ),
    embedding_dim=st_model.get_sentence_embedding_dimension(),
)
```

Then pass it to `load_demo_splits()`.

In [ ]:
# Uncomment and edit to use your own data:

# my_train, my_holdout = load_demo_splits(
#     val_file="path/to/my_train.jsonl",
#     holdout_file="path/to/my_holdout.jsonl",
#     feature_service=fs,        # or custom_fs from above
# )
#
# cfg = DemoConfig(output_dir="my_results", n_seeds=5)
# out = run_scenario_1(cfg, my_train, my_holdout)
# display(Image(filename=str(out)))

print("Edit this cell to load your own JSONL data and re-run!")